# 02. Empirical eigenmodes

This notebook computes the monthly sectoral correlation matrix, leading eigenvectors, PC score series, and PC score ACFs.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import binom, betabinom, norm, rankdata, spearmanr, kendalltau
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(".")
DATA = ROOT / "data"
PDATA = ROOT / "pdata"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
for directory in [DATA, PDATA, FIGURES, TABLES]:
    directory.mkdir(exist_ok=True)

RNG = np.random.default_rng(20260618)
plt.style.use("default")


SECTOR_COLORS = {
    "Basic Materials": "#4C78A8",
    "Capital Goods": "#F58518",
    "Consumer": "#54A24B",
    "Energy": "#E45756",
    "Finance": "#72B7B2",
    "Healthcare": "#B279A2",
    "Technology": "#FF9DA6",
    "Utilities": "#9D755D",
}


def read_sample():
    df = pd.read_csv(DATA / "SAMPLE.csv", parse_dates=["date"])
    expected = ["date", "sector", "obligors", "defaulted"]
    if list(df.columns) != expected:
        raise ValueError(f"Expected columns {expected}, got {list(df.columns)}")
    df = df.sort_values(["date", "sector"]).reset_index(drop=True)
    if (df["obligors"] <= 0).any():
        raise ValueError("obligors must be positive")
    if ((df["defaulted"] < 0) | (df["defaulted"] > df["obligors"])).any():
        raise ValueError("defaulted must lie between zero and obligors")
    return df


def empirical_logit(defaulted, obligors, smooth=0.5):
    defaulted = np.asarray(defaulted, dtype=float)
    obligors = np.asarray(obligors, dtype=float)
    return np.log((defaulted + smooth) / (obligors - defaulted + smooth))


def panel_matrices(df):
    dates = pd.Index(sorted(df["date"].unique()), name="date")
    sectors = pd.Index(sorted(df["sector"].unique()), name="sector")
    dmat = (
        df.pivot(index="date", columns="sector", values="defaulted")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    nmat = (
        df.pivot(index="date", columns="sector", values="obligors")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    x = pd.DataFrame(
        empirical_logit(dmat.to_numpy(), nmat.to_numpy()),
        index=dates,
        columns=sectors,
    )
    rates = dmat / nmat
    return dates, sectors, dmat, nmat, x, rates


def aggregate_k_month(df, k):
    wide_dates = pd.Index(sorted(df["date"].unique()))
    block_lookup = {date: i // k for i, date in enumerate(wide_dates)}
    tmp = df.copy()
    tmp["block"] = tmp["date"].map(block_lookup)
    tmp["block_start"] = tmp["block"].map(lambda b: wide_dates[b * k])
    out = (
        tmp.groupby(["block", "block_start", "sector"], as_index=False)
        .agg(obligors=("obligors", "sum"), defaulted=("defaulted", "sum"))
    )
    out = out.rename(columns={"block_start": "date"})
    out["k_month"] = k
    return out[["date", "sector", "k_month", "obligors", "defaulted"]]


def fit_common_eps_model(df, R=2):
    dates, sectors, dmat, nmat, x, rates = panel_matrices(df)
    X = x.to_numpy()
    intercept = X.mean(axis=0)
    Xc = X - intercept
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    loadings = eigvecs[:, :R]
    scores = Xc @ loadings
    phis = []
    innov_sd = []
    for r in range(R):
        y = scores[1:, r]
        z = scores[:-1, r]
        denom = float(np.dot(z, z))
        phi = float(np.dot(z, y) / denom) if denom > 1e-12 else 0.0
        phi = float(np.clip(phi, -0.98, 0.98))
        resid = y - phi * z
        phis.append(phi)
        innov_sd.append(float(np.sqrt(np.mean(resid**2) + 1e-8)))
    fitted_centered = scores @ loadings.T
    residual = Xc - fitted_centered
    sigma_eps_common = float(np.sqrt(np.mean(residual**2) + 1e-8))
    eta_hat = intercept + fitted_centered
    p_hat = expit(eta_hat)
    ll = float(binom.logpmf(dmat.to_numpy(), nmat.to_numpy(), p_hat).sum())
    n_params = len(sectors) + R * (len(sectors) + 2) + 1
    aic = float(2 * n_params - 2 * ll)
    return {
        "dates": list(dates),
        "sectors": list(sectors),
        "defaulted": dmat,
        "obligors": nmat,
        "logit": x,
        "rates": rates,
        "intercept": intercept,
        "eigvals": eigvals,
        "loadings": loadings,
        "scores": scores,
        "phis": np.array(phis),
        "innov_sd": np.array(innov_sd),
        "sigma_eps_common": sigma_eps_common,
        "eta_hat": eta_hat,
        "p_hat": p_hat,
        "log_likelihood": ll,
        "aic": aic,
    }


def acf_values(x, max_lag=24):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.dot(x, x)
    if denom <= 1e-12:
        return np.zeros(max_lag + 1)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.array(vals)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

In [ ]:
df = read_sample()
model = fit_common_eps_model(df, R=4)
sectors = list(model["sectors"])
eigvals = model["eigvals"]
shares = eigvals / eigvals.sum()

corr = model["logit"].corr()
corr.to_csv(PDATA / "monthly_sector_correlation.csv")

eigenmode_summary = pd.DataFrame({
    "mode": np.arange(1, len(eigvals) + 1),
    "factor_interpretation": ["market-wide default-risk factor", "sector-rotation factor"] + ["higher-order diagnostic"] * (len(eigvals) - 2),
    "eigenvalue": eigvals,
    "variance_share": shares,
    "cumulative_share": np.cumsum(shares),
})
eigenmode_summary.to_csv(PDATA / "eigenmode_summary.csv", index=False)

sector_rates = df.assign(default_rate=df["defaulted"] / df["obligors"])
table3 = (
    sector_rates.groupby("sector")
    .agg(
        months=("date", "nunique"),
        obligor_months=("obligors", "sum"),
        defaults=("defaulted", "sum"),
        mean_monthly_default_rate=("default_rate", "mean"),
        sd_monthly_default_rate=("default_rate", "std"),
    )
    .reindex(sectors)
    .reset_index()
)
table3["lambda_F1_market_wide"] = model["loadings"][:, 0]
table3["lambda_F2_sector_rotation"] = model["loadings"][:, 1]
table3["sector_logit_sd"] = model["logit"].std(axis=0).to_numpy()
table3.to_csv(TABLES / "table3_sector_level_summary_statistics.csv", index=False)

loadings_df = pd.DataFrame(model["loadings"][:, :4], index=sectors, columns=[f"PC{r}" for r in range(1, 5)])
loadings_df.to_csv(PDATA / "eigenmodes.csv")
scores_df = pd.DataFrame(model["scores"][:, :4], index=model["dates"], columns=[f"PC{r}" for r in range(1, 5)])
scores_df.index.name = "date"
scores_df.to_csv(PDATA / "pc_scores.csv")

acf_rows = []
for r in range(4):
    for lag, value in enumerate(acf_values(model["scores"][:, r], 24)):
        acf_rows.append({"mode": f"PC{r+1}", "lag_months": lag, "acf": value})
pd.DataFrame(acf_rows).to_csv(PDATA / "pc_score_acf.csv", index=False)
table3

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
x = np.arange(len(sectors))
width = 0.36
ax.bar(x - width / 2, loadings_df["PC1"], width, label="F1: market-wide default-risk", color="#4C78A8")
ax.bar(x + width / 2, loadings_df["PC2"], width, label="F2: sector-rotation", color="#F58518")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(sectors, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Eigenvector loading")
ax.set_title("Figure 2. Leading empirical eigenvectors")
ax.legend(fontsize=8)
savefig(FIGURES / "figure2_leading_empirical_eigenvectors.png")

In [ ]:
acf_df = pd.read_csv(PDATA / "pc_score_acf.csv")
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for mode, color in zip(["PC1", "PC2", "PC3", "PC4"], ["#4C78A8", "#F58518", "#54A24B", "#E45756"]):
    g = acf_df[acf_df["mode"] == mode]
    ax.plot(g["lag_months"], g["acf"], marker="o", ms=3, label=mode, color=color)
ax.axhline(0.0, color="black", lw=0.8)
ax.set_title("Figure 3. Empirical PC score ACFs")
ax.set_xlabel("Lag, months")
ax.set_ylabel("ACF")
ax.legend(ncol=2)
savefig(FIGURES / "figure3_empirical_pc_score_acfs.png")